# Visualise Ancestry 

In [1]:
# visualize_ancestry.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import itertools
import re
import os
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors
import matplotlib
from typing import List, Tuple, Dict, Any, Optional, Literal # If you want to keep type hints

In [2]:
def get_unique_ancestries(locus_data_df):
    ancestries = set()
    for chromatid in ['chromatid1', 'chromatid2']:
        pop_col = f'ancestry_{chromatid}_pop'
        founder_col = f'ancestry_{chromatid}_founder_id'
        filtered = locus_data_df[[pop_col, founder_col]].dropna()
        for _, row in filtered.iterrows():
            ancestries.add((row[pop_col], int(row[founder_col])))
    return ancestries

def generate_color_map_for_ancestries(unique_ancestries):
    # Use the new matplotlib colormaps API
    cmap = matplotlib.colormaps['tab20'].resampled(len(unique_ancestries))
    color_map = {}
    for i, anc in enumerate(sorted(unique_ancestries)):
        rgba = cmap(i)
        color_map[anc] = mcolors.to_hex(rgba)
    return color_map


In [3]:
def plot_diploid_chromosome_ancestry(
    individual_id: int,
    diploid_chr_id: int,
    generation_label: str,
    locus_data_df: pd.DataFrame,
    marker_height: float = 0.8,
    spacing: float = 1.2,
    save_filename: Optional[str] = None,
    max_loci_per_row: int = 80,
    row_height: float = 3.0
):
    target_data = locus_data_df[
        (locus_data_df['individual_id'] == individual_id) &
        (locus_data_df['diploid_chr_id'] == diploid_chr_id) &
        (locus_data_df['generation'] == generation_label)
    ].sort_values(by='locus_position').reset_index(drop=True)

    if target_data.empty:
        print(f"No data found for Individual ID {individual_id}, Chromosome {diploid_chr_id} in generation {generation_label}.")
        return

    num_loci = len(target_data)
    unique_ancestries = get_unique_ancestries(locus_data_df)
    ancestry_colors = generate_color_map_for_ancestries(unique_ancestries)

    # Determine number of rows to split loci if too many
    num_rows = (num_loci + max_loci_per_row - 1) // max_loci_per_row
    loci_per_row = min(num_loci, max_loci_per_row)

    fig_width = max(12, loci_per_row * 0.25)  # increase spacing per locus a bit
    fig_height = row_height * num_rows

    fig, axs = plt.subplots(num_rows, 1, figsize=(fig_width, fig_height), squeeze=False)
    axs = axs.flatten()

    legend_elements = []
    used_ancestries = set()

    for row_idx in range(num_rows):
        ax = axs[row_idx]
        start_idx = row_idx * max_loci_per_row
        end_idx = min(start_idx + max_loci_per_row, num_loci)
        row_data = target_data.iloc[start_idx:end_idx]

        ax.set_title(f"Individual {individual_id}, Chromosome {diploid_chr_id} ({generation_label})\n"
                     "Ancestral Origin", fontsize=14 if row_idx == 0 else 10)
        ax.set_xlabel("Locus Position", fontsize=12)
        ax.set_ylabel("Chromatid", fontsize=12)
        ax.set_yticks([-spacing/2, spacing/2])
        ax.set_yticklabels(['Chromatid 2', 'Chromatid 1'])

        tick_interval = max(1, loci_per_row // 20)
        ticks = np.arange(0, loci_per_row, tick_interval)
        labels = ticks + start_idx  # shift labels to match loci indices

        ax.set_xticks(ticks)
        ax.set_xticklabels(labels)

        ax.set_xlim(-0.5, loci_per_row - 0.5)
        ax.set_ylim(-spacing, spacing)
        ax.set_aspect('auto', adjustable='box')
        ax.axis('off')

        ax.plot([-0.5, loci_per_row - 0.5], [spacing/2, spacing/2], color='black', linewidth=1.5, zorder=1)
        ax.plot([-0.5, loci_per_row - 0.5], [-spacing/2, -spacing/2], color='black', linewidth=1.5, zorder=1)

        for i, row in row_data.iterrows():
            local_idx = i - start_idx
            for chromatid, y_offset in zip(['chromatid1', 'chromatid2'], [spacing/2, -spacing/2]):
                anc_pop = row[f'ancestry_{chromatid}_pop']
                founder_id = row[f'ancestry_{chromatid}_founder_id']
                if pd.isna(anc_pop) or pd.isna(founder_id):
                    continue
                ancestry = (anc_pop, int(founder_id))
                color = ancestry_colors.get(ancestry, 'gray')

                rect = patches.Rectangle(
                    (local_idx - 0.4, y_offset - marker_height/2),
                    0.8,
                    marker_height,
                    facecolor=color,
                    edgecolor='none',
                    linewidth=0,
                    zorder=2
                )
                ax.add_patch(rect)

                # Calculate text color contrast
                try:
                    import matplotlib.colors as mcolors
                    rgb = mcolors.to_rgb(color)
                    text_color = 'black' if np.mean(rgb) > 0.5 else 'white'
                except Exception:
                    text_color = 'black'

                ax.text(
                    local_idx,
                    y_offset,
                    str(founder_id),
                    ha='center',
                    va='center',
                    fontsize=7,
                    color=text_color,
                    zorder=3
                )

                if ancestry not in used_ancestries:
                    legend_elements.append(Line2D([0], [0], marker='s', color='w', label=f'{anc_pop}_{founder_id}',
                                                  markerfacecolor=color, markersize=8))
                    used_ancestries.add(ancestry)

    if legend_elements:
        axs[-1].legend(
            handles=legend_elements,
            title="Ancestral Origin",
            loc='upper right',
            bbox_to_anchor=(1.05, 1),
            borderaxespad=0.0,
            frameon=False,
            fontsize=9,
            title_fontsize=10
        )

    fig.subplots_adjust(right=0.8, hspace=0.4)

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Chromosome plot saved to: {save_filename}")
        plt.close(fig)  # Close to free memory
    else:
        return fig

In [4]:
# Example 1: List IDs in a specific generation
import pandas as pd

# Load the locus-level genotype data with ancestry info
locus_level_df = pd.read_csv('output_data/dataframes/locus_level_genotypes_with_ancestry.csv')

generation_to_check = 'F5'  # Change this to any generation you want to inspect
print(f"\n--- Individual IDs in Generation '{generation_to_check}' ---")
ids_in_generation = locus_level_df.loc[
    locus_level_df['generation'] == generation_to_check,
    'individual_id'
].unique().tolist()

if ids_in_generation:
    print(f"Total individuals in '{generation_to_check}': {len(ids_in_generation)}")
    print("First 10 IDs (if available):", ids_in_generation[:10])
    print("Last 10 IDs (if available):", ids_in_generation[-10:])
else:
    print(f"No individuals found for generation '{generation_to_check}'.")


--- Individual IDs in Generation 'F5' ---
Total individuals in 'F5': 100
First 10 IDs (if available): [600, 601, 602, 603, 604, 605, 606, 607, 608, 609]
Last 10 IDs (if available): [690, 691, 692, 693, 694, 695, 696, 697, 698, 699]


In [5]:
if __name__ == "__main__":
    print("\nStarting Chr Ancestry Visualisation")

    # Plot a SPECIFIC individual by its ID
    # NOTE: Make sure this ID actually exists in your data for the specified generation!
    # You might get this ID from the list printed in Example 1, or from other analysis.
    specific_id_to_plot = 602  # <--- CHANGE THIS to an ID you know exists!
    specific_gen_for_id = 'F5'  # <--- CHANGE THIS to the generation of that specific ID!
    specific_chr_to_plot = 2  # <--- CHANGE THIS to the chromosome pair (1 or 2)

    plots_directory = 'output_data/plots'  # or wherever you want

    # Make sure the directory exists
    os.makedirs(plots_directory, exist_ok=True)

    # Check if the specific individual exists in the DataFrame before attempting to plot
    if not locus_level_df[
        (locus_level_df['individual_id'] == specific_id_to_plot) &
        (locus_level_df['generation'] == specific_gen_for_id) &
        (locus_level_df['diploid_chr_id'] == specific_chr_to_plot)
    ].empty:

        plot_save_path_specific_ind = os.path.join(
            plots_directory,
            f"specific_ind_{specific_id_to_plot}_gen_{specific_gen_for_id}_chr_{specific_chr_to_plot}_ancestry.png"
        )

        print(f"\nPlotting specific individual (ID: {specific_id_to_plot}) from generation '{specific_gen_for_id}', Chromosome {specific_chr_to_plot}")
        plot_diploid_chromosome_ancestry(
            individual_id=specific_id_to_plot,
            diploid_chr_id=specific_chr_to_plot,
            generation_label=specific_gen_for_id,
            locus_data_df=locus_level_df,
            save_filename=plot_save_path_specific_ind
        )
    else:
        print(f"\nSkipping plot for specific individual {specific_id_to_plot}: Data not found for this ID, generation, and chromosome combination.")



Starting Chr Ancestry Visualisation

Plotting specific individual (ID: 602) from generation 'F5', Chromosome 2
Chromosome plot saved to: output_data/plots\specific_ind_602_gen_F5_chr_2_ancestry.png
